In [1]:
with open('data/A_IUSG51WION121200_C_WIIX_20251012120000.bin', 'rb') as f:
    content = f.read()
    print(content[:20])  # Look at the first 20 bytes

b'IUSG18 WION 121200\r\n'


In [2]:
import eccodes

def read_bufr(file_path):
    with open(file_path, 'rb') as f:
        while True:
            # Radiosonde data is often wrapped in BUFR messages
            msgid = eccodes.codes_bufr_new_from_file(f)
            if msgid is None:
                break
            
            # You must "expand" the descriptors to see the values
            eccodes.codes_set(msgid, 'unpack', 1)
            
            # Example: Get temperature and pressure
            # These keys vary depending on the BUFR edition
            try:
                temp = eccodes.codes_get_array(msgid, 'airTemperature')
                press = eccodes.codes_get_array(msgid, 'pressure')
                print(f"Temperature: {temp}")
                print(f"Pressure: {press}")
            except eccodes.KeyValueNotFoundError:
                print("Key not found in this message")
            
            eccodes.codes_release(msgid)

read_bufr('data/A_IUSG51WION121200_C_WIIX_20251012120000.bin')

Temperature: [300.75 299.46 299.43 ... 194.4  194.52 194.68]
Pressure: [100600. 100540. 100440. ...   6020.   6000.   5980.]


In [7]:
import eccodes
import pandas as pd
import numpy as np

def extract_radiosonde_data(file_path):
    with open(file_path, 'rb') as f:
        msgid = eccodes.codes_bufr_new_from_file(f)
        if msgid is None:
            return None
        
        eccodes.codes_set(msgid, 'unpack', 1)
        
        # 1. Get the profile data (The long arrays)
        # These are usually the vertical sounding parameters
        profile_keys = {
            'pressure': 'pressure_Pa',
            'airTemperature': 'temp_K',
            'dewpointTemperature': 'dewpoint_K',
            'windDirection': 'wind_dir_deg',
            'windSpeed': 'wind_speed_mps',
            'nonCoordinateGeopotentialHeight': 'height_m'
        }
        
        data = {}
        lengths = []
        
        for bufr_key, df_name in profile_keys.items():
            try:
                val = eccodes.codes_get_array(msgid, bufr_key)
                data[df_name] = val
                lengths.append(len(val))
            except eccodes.KeyValueNotFoundError:
                continue

        eccodes.codes_release(msgid)

        # 2. Check for length consistency
        if not lengths:
            return pd.DataFrame()
            
        max_len = max(lengths)
        
        # Pad or trim arrays to match the longest one (usually pressure)
        # Or simply only keep keys that match the max length
        clean_data = {}
        for k, v in data.items():
            if len(v) == max_len:
                clean_data[k] = v
            else:
                print(f"Skipping {k}: length {len(v)} does not match max length {max_len}")

        df = pd.DataFrame(clean_data)

        # 3. Unit Conversions (Safe check in case keys were skipped)
        if 'temp_K' in df.columns:
            df['temp_C'] = df['temp_K'] - 273.15
        if 'dewpoint_K' in df.columns:
            df['dewpoint_C'] = df['dewpoint_K'] - 273.15
        if 'pressure_Pa' in df.columns:
            df['pressure_hPa'] = df['pressure_Pa'] / 100.0
            
        return df

df = extract_radiosonde_data('data/A_IUSG51WION121200_C_WIIX_20251012120000.bin')
print(df.head())

   pressure_Pa  temp_K  dewpoint_K  wind_dir_deg  wind_speed_mps  height_m  \
0     100600.0  300.75      297.68           214             4.8        30   
1     100540.0  299.46      298.20           213             4.9        36   
2     100440.0  299.43      298.16           213             5.0        44   
3     100320.0  299.36      298.12           213             5.1        55   
4     100240.0  299.30      298.10           213             5.2        63   

   temp_C  dewpoint_C  pressure_hPa  
0   27.60       24.53        1006.0  
1   26.31       25.05        1005.4  
2   26.28       25.01        1004.4  
3   26.21       24.97        1003.2  
4   26.15       24.95        1002.4  
